# Incident-risk analysis from simulated traffic exposure

This notebook combines Emilia-Romagna road-accident statistical tables (`2022.xlsx`, `2023.xlsx`, `2024.xlsx`) with simulated link-level traffic outputs for weekdays and holidays.

The goal is not to geolocate individual crashes, because the regional Excel tables are aggregated. Instead, the notebook builds **exposure-normalized safety indicators**:

- hourly traffic exposure vs hourly accidents;
- exposure-normalized accident intensity by hour;
- weekday/holiday comparison when both simulations are available;
- seasonal exposure vs accident seasonality;
- hazard-weighted traffic exposure maps over the simulated road network.

Run this notebook from the repository root, or edit the paths in the configuration cell.

In [19]:
# Common paper palette and style, aligned with notebooks 01--04.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm

PALETTE = {
    "src_dest": "#E7D1D1",
    "rnd_rnd": "#DCC7DF",
    "rnd_dest": "#C4DADD",
    "src_rnd": "#C8DEC8",
    "roads_main": "#8D8D8D",
    "roads_minor": "#CFCFCF",
    "text": "#2F2F2F",
    "background": "#FAFAFA",
    "grid": "#DDDDDD",
}

ACCIDENT_COLORS = {
    "incidents": "#B84A62",
    "deaths": "#252525",
    "injured": "#6B8FA6",
    "weekday": "#8D8D8D",
    "holiday": "#D08C60",
    "risk": "#B84A62",
    "exposure": "#2A9D8F",
}

paper_cmap = LinearSegmentedColormap.from_list(
    "paper_cmap",
    ["#F7F7F7", "#DAD7CD", "#A3B18A", "#588157", "#3A5A40"],
)

risk_cmap = LinearSegmentedColormap.from_list(
    "risk_cmap",
    ["#F7F7F7", "#F4D6CC", "#D08C60", "#B84A62", "#6D213C"],
)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": PALETTE["background"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "text.color": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "axes.titlesize": 22,
    "axes.labelsize": 18,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,
    "font.family": "DejaVu Sans",
    "savefig.dpi": 300,
})


In [20]:
# Imports, paths and output utility.
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    from shapely import wkt
except Exception as exc:
    gpd = None
    wkt = None
    print(f"Geospatial stack unavailable: {exc}")

try:
    import contextily as cx
except Exception:
    cx = None

# ---------------------------------------------------------------------
# Edit these paths if needed.
# ---------------------------------------------------------------------
INCIDENT_DATA_DIR = Path("/home/fede/code/covid_hep/work_incidenti/data")
SIM_DATA_DIR = Path("/home/fede/code/covid_hep/work_iconsultin/data2")

# Sandbox/local fallback, useful only when testing this exported notebook elsewhere.
if not INCIDENT_DATA_DIR.exists():
    INCIDENT_DATA_DIR = Path("/mnt/data")
if not SIM_DATA_DIR.exists():
    SIM_DATA_DIR = Path("/mnt/data")

WEEKDAY_DIR = SIM_DATA_DIR / "feriali"
HOLIDAY_DIR = SIM_DATA_DIR / "festivi"

# If folders are absent, the loader below also falls back to CSVs directly inside SIM_DATA_DIR.
OUTPUT_DIR = Path("article/figures_incident_risk")
if not Path("article").exists():
    OUTPUT_DIR = Path("figures_incident_risk")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INCIDENT_FILES = {
    2022: INCIDENT_DATA_DIR / "2022.xlsx",
    2023: INCIDENT_DATA_DIR / "2023.xlsx",
    2024: INCIDENT_DATA_DIR / "2024.xlsx",
}
# Some regional downloads use a longer 2024 filename.
if not INCIDENT_FILES[2024].exists():
    alt_2024 = INCIDENT_DATA_DIR / "08-emila-romagna-tavole-2024.xlsx"
    if alt_2024.exists():
        INCIDENT_FILES[2024] = alt_2024

HOUR_TICKS = np.arange(0, 24, 3)
HOUR_LABELS = [f"{h:02d}:00" for h in range(24)]


def save_fig_local(fig, stem):
    """Save figures as PNG and PDF, following the convention of notebooks 01--04."""
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    print(f"Saved: {pdf}")


def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="y", color=PALETTE["grid"], linewidth=0.8, alpha=0.75, zorder=0)
    ax.set_axisbelow(True)


## 1. Load regional accident tables

The Excel workbooks are aggregated statistical tables. The notebook extracts the sheets needed for exposure-based analysis:

- `Tavola 7`: accidents by month;
- `Tavola 8`: accidents by day of week;
- `Tavola 9`: accidents by Time;
- `Tavola 5`: accidents by road category.

In [21]:
# Accident-table extractors.

def _as_number(x):
    return pd.to_numeric(x, errors="coerce")


def _parse_hour_label(value):
    """Convert ISTAT hour labels ('1', '1a ora', '24a "') to 0--23."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    match = re.search(r"(\d+)", s)
    if not match:
        return np.nan
    h = int(match.group(1))
    if 1 <= h <= 24:
        return h - 1
    if 0 <= h <= 23:
        return h
    return np.nan


def extract_tavola9_hourly(path, year):
    df = pd.read_excel(path, sheet_name="Tavola 9", header=None)
    records = []
    for _, row in df.iterrows():
        hour = _parse_hour_label(row.iloc[1] if len(row) > 1 else np.nan)
        incidents = _as_number(row.iloc[2] if len(row) > 2 else np.nan)
        deaths = _as_number(row.iloc[3] if len(row) > 3 else np.nan)
        injured = _as_number(row.iloc[4] if len(row) > 4 else np.nan)
        if pd.notna(hour) and pd.notna(incidents):
            records.append({
                "year": year,
                "hour": int(hour),
                "incidents": float(incidents),
                "deaths": float(deaths) if pd.notna(deaths) else np.nan,
                "injured": float(injured) if pd.notna(injured) else np.nan,
            })
    out = pd.DataFrame(records)
    out = out[out["hour"].between(0, 23)].sort_values(["year", "hour"])
    return out


def extract_tavola8_weekday(path, year):
    df = pd.read_excel(path, sheet_name="Tavola 8", header=None)
    day_order = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]
    records = []
    for _, row in df.iterrows():
        day = row.iloc[1] if len(row) > 1 else None
        if isinstance(day, str) and day.strip() in day_order:
            records.append({
                "year": year,
                "day": day.strip(),
                "day_index": day_order.index(day.strip()),
                "incidents": float(_as_number(row.iloc[2])),
                "deaths": float(_as_number(row.iloc[3])),
                "injured": float(_as_number(row.iloc[4])),
            })
    return pd.DataFrame(records).sort_values(["year", "day_index"])


def extract_tavola7_monthly(path, year):
    df = pd.read_excel(path, sheet_name="Tavola 7", header=None)
    month_order = [
        "Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno",
        "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre",
    ]
    records = []
    for _, row in df.iterrows():
        month = row.iloc[1] if len(row) > 1 else None
        if isinstance(month, str) and month.strip() in month_order:
            mi = month_order.index(month.strip()) + 1
            season = {
                12: "winter", 1: "winter", 2: "winter",
                3: "spring", 4: "spring", 5: "spring",
                6: "summer", 7: "summer", 8: "summer",
                9: "autumn", 10: "autumn", 11: "autumn",
            }[mi]
            records.append({
                "year": year,
                "month": month.strip(),
                "month_index": mi,
                "season": season,
                "incidents": float(_as_number(row.iloc[2])),
                "deaths": float(_as_number(row.iloc[3])),
                "injured": float(_as_number(row.iloc[4])),
            })
    return pd.DataFrame(records).sort_values(["year", "month_index"])


def extract_tavola5_road_category(path, year):
    df = pd.read_excel(path, sheet_name="Tavola 5", header=None)
    wanted = {"Strade urbane", "Autostrade e raccordi", "Altre strade (c)", "Totale"}
    records = []
    for _, row in df.iterrows():
        category = row.iloc[1] if len(row) > 1 else None
        if isinstance(category, str) and category.strip() in wanted:
            records.append({
                "year": year,
                "road_category": category.strip().replace(" (c)", ""),
                "incidents": float(_as_number(row.iloc[2])),
                "deaths": float(_as_number(row.iloc[3])),
                "injured": float(_as_number(row.iloc[4])),
            })
    return pd.DataFrame(records)


def load_accident_tables(files):
    hourly, weekdays, monthly, categories = [], [], [], []
    for year, path in files.items():
        if not path.exists():
            print(f"Missing accident workbook for {year}: {path}")
            continue
        print(f"Loading accident workbook {year}: {path}")
        hourly.append(extract_tavola9_hourly(path, year))
        weekdays.append(extract_tavola8_weekday(path, year))
        monthly.append(extract_tavola7_monthly(path, year))
        categories.append(extract_tavola5_road_category(path, year))
    return {
        "hourly": pd.concat(hourly, ignore_index=True) if hourly else pd.DataFrame(),
        "weekday": pd.concat(weekdays, ignore_index=True) if weekdays else pd.DataFrame(),
        "monthly": pd.concat(monthly, ignore_index=True) if monthly else pd.DataFrame(),
        "road_category": pd.concat(categories, ignore_index=True) if categories else pd.DataFrame(),
    }

acc = load_accident_tables(INCIDENT_FILES)
for name, df in acc.items():
    print(f"{name:>14}: {df.shape}")
    display(df.head())


Loading accident workbook 2022: /home/fede/code/covid_hep/work_incidenti/data/2022.xlsx
Loading accident workbook 2023: /home/fede/code/covid_hep/work_incidenti/data/2023.xlsx
Loading accident workbook 2024: /home/fede/code/covid_hep/work_incidenti/data/2024.xlsx
        hourly: (72, 5)


,year,hour,incidents,deaths,injured
0,2022,0,264.0,8.0,403.0
1,2022,1,173.0,2.0,247.0
2,2022,2,115.0,7.0,172.0
3,2022,3,109.0,8.0,160.0
4,2022,4,118.0,6.0,203.0


       weekday: (21, 6)


,year,day,day_index,incidents,deaths,injured
0,2022,Lunedì,0,2409.0,37.0,3057.0
1,2022,Martedì,1,2513.0,36.0,3184.0
2,2022,Mercoledì,2,2509.0,39.0,3118.0
3,2022,Giovedì,3,2494.0,48.0,3212.0
4,2022,Venerdì,4,2712.0,38.0,3486.0


       monthly: (36, 7)


,year,month,month_index,season,incidents,deaths,injured
0,2022,Gennaio,1,winter,981.0,21.0,1248.0
1,2022,Febbraio,2,winter,1108.0,17.0,1400.0
2,2022,Marzo,3,spring,1251.0,16.0,1590.0
3,2022,Aprile,4,spring,1278.0,30.0,1686.0
4,2022,Maggio,5,spring,1694.0,19.0,2205.0


 road_category: (12, 5)


,year,road_category,incidents,deaths,injured
0,2022,Strade urbane,11800.0,127.0,14503.0
1,2022,Autostrade e raccordi,840.0,38.0,1527.0
2,2022,Altre strade,4039.0,146.0,5646.0
3,2022,Totale,16679.0,311.0,21676.0
4,2023,Strade urbane,11945.0,118.0,14645.0


## 2. Load simulated traffic exposure

The simulation tables are expected in:

- `.../data2/feriali/feriali.csv` for weekdays;
- `.../data2/festivi/festivi.csv` for holidays.

The loader also accepts individual simulation CSVs with columns `poly_lid, datetime, counter` and aggregates them automatically.

In [22]:
# Simulation loaders.

def read_csv_flexible(path):
    """Read comma- or semicolon-separated simulation CSVs."""
    path = Path(path)
    # Try Python engine with automatic delimiter detection first.
    try:
        df = pd.read_csv(path, sep=None, engine="python")
    except Exception:
        df = pd.read_csv(path)
    # Some files can be parsed as one column; retry with the other separator.
    if df.shape[1] == 1:
        for sep in [";", ","]:
            try:
                cand = pd.read_csv(path, sep=sep)
                if cand.shape[1] > 1:
                    return cand
            except Exception:
                pass
    return df


def _load_aggregate_simulation(path, label):
    df = read_csv_flexible(path)
    if "poly_lid" not in df.columns or "counter" not in df.columns:
        raise KeyError(f"Expected columns poly_lid and counter in {path}. Found: {list(df.columns)}")
    df["source"] = label
    df["counter"] = pd.to_numeric(df["counter"], errors="coerce").fillna(0.0)
    if "hour" in df.columns:
        df["hour"] = pd.to_numeric(df["hour"], errors="coerce").astype("Int64")
    elif "datetime" in df.columns:
        dt = pd.to_datetime(df["datetime"], errors="coerce")
        df["hour"] = dt.dt.hour.astype("Int64")
    else:
        raise KeyError(f"No hour or datetime column found in {path}")
    if "season" not in df.columns:
        df["season"] = "all"
    return df


def _load_individual_simulation_files(folder, label):
    folder = Path(folder)
    files = sorted(folder.glob("*.csv")) if folder.exists() else []
    files = [p for p in files if p.name.lower() not in {"feriali.csv", "festivi.csv"}]
    # Sandbox/testing fallback: if no weekday folder exists, use CSVs directly in SIM_DATA_DIR.
    # Do not apply this fallback to holidays, otherwise weekday files could be duplicated as holidays.
    if not files and label == "weekday":
        files = [p for p in sorted(SIM_DATA_DIR.glob("*.csv")) if p.name.lower() not in {"feriali.csv", "festivi.csv"}]
    pieces = []
    for p in files:
        df = read_csv_flexible(p)
        if not {"poly_lid", "datetime", "counter"}.issubset(df.columns):
            continue
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
        df["hour"] = df["datetime"].dt.hour.astype("Int64")
        df["counter"] = pd.to_numeric(df["counter"], errors="coerce").fillna(0.0)
        df["season"] = "all"
        df["source"] = label
        pieces.append(df[["poly_lid", "hour", "counter", "season", "source"]])
    if not pieces:
        return pd.DataFrame()
    return pd.concat(pieces, ignore_index=True)


def load_simulation(kind):
    """Load weekday or holiday exposure tables."""
    assert kind in {"weekday", "holiday"}
    folder = WEEKDAY_DIR if kind == "weekday" else HOLIDAY_DIR
    aggregate_name = "feriali.csv" if kind == "weekday" else "festivi.csv"
    aggregate_candidates = [folder / aggregate_name, SIM_DATA_DIR / aggregate_name]
    for candidate in aggregate_candidates:
        if candidate.exists():
            print(f"Loading {kind} aggregate simulation: {candidate}")
            return _load_aggregate_simulation(candidate, kind)
    print(f"No aggregate {kind} file found. Trying individual CSV files in {folder}.")
    return _load_individual_simulation_files(folder, kind)


weekday_sim = load_simulation("weekday")
holiday_sim = load_simulation("holiday")

print("weekday_sim", weekday_sim.shape)
print("holiday_sim", holiday_sim.shape)
display(weekday_sim.head())
if not holiday_sim.empty:
    display(holiday_sim.head())
else:
    print("Holiday simulation not found. Holiday-specific panels will be skipped or shown as unavailable.")


Loading weekday aggregate simulation: /home/fede/code/covid_hep/work_iconsultin/data2/feriali/feriali.csv
Loading holiday aggregate simulation: /home/fede/code/covid_hep/work_iconsultin/data2/festivi/festivi.csv
weekday_sim (373152, 7)
holiday_sim (373152, 7)


,Unnamed: 0,poly_lid,season,hour,counter,geometry,source
0,0,0,autumn,0,0,"LINESTRING (11.026845 44.438334, 11.026428 44....",weekday
1,1,0,autumn,1,0,"LINESTRING (11.026845 44.438334, 11.026428 44....",weekday
2,2,0,autumn,2,3,"LINESTRING (11.026845 44.438334, 11.026428 44....",weekday
3,3,0,autumn,3,2,"LINESTRING (11.026845 44.438334, 11.026428 44....",weekday
4,4,0,autumn,4,1,"LINESTRING (11.026845 44.438334, 11.026428 44....",weekday


,Unnamed: 0,poly_lid,season,hour,counter,geometry,source
0,0,0,autumn,0,0,"LINESTRING (11.026845 44.438334, 11.026428 44....",holiday
1,1,0,autumn,1,0,"LINESTRING (11.026845 44.438334, 11.026428 44....",holiday
2,2,0,autumn,2,5,"LINESTRING (11.026845 44.438334, 11.026428 44....",holiday
3,3,0,autumn,3,3,"LINESTRING (11.026845 44.438334, 11.026428 44....",holiday
4,4,0,autumn,4,4,"LINESTRING (11.026845 44.438334, 11.026428 44....",holiday


In [23]:
# Build exposure tables and optional GeoDataFrames.

def hourly_exposure(sim_df):
    if sim_df.empty:
        return pd.Series(dtype=float)
    out = sim_df.dropna(subset=["hour"]).groupby("hour")["counter"].sum().reindex(range(24), fill_value=0.0)
    out.index = out.index.astype(int)
    return out


def seasonal_exposure(sim_df):
    if sim_df.empty or "season" not in sim_df.columns:
        return pd.DataFrame()
    out = sim_df.groupby("season")["counter"].sum().reset_index()
    return out


def link_hour_matrix(sim_df):
    if sim_df.empty:
        return pd.DataFrame()
    mat = (
        sim_df.dropna(subset=["hour"])
        .pivot_table(index="poly_lid", columns="hour", values="counter", aggfunc="sum", fill_value=0.0)
        .reindex(columns=range(24), fill_value=0.0)
    )
    return mat


def link_geometries(sim_df):
    if sim_df.empty or "geometry" not in sim_df.columns or gpd is None or wkt is None:
        return None
    geom_df = sim_df[["poly_lid", "geometry"]].dropna().drop_duplicates("poly_lid").copy()
    if geom_df.empty:
        return None
    geom_df["geometry"] = geom_df["geometry"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
    return gpd.GeoDataFrame(geom_df, geometry="geometry", crs="EPSG:4326")

weekday_hourly = hourly_exposure(weekday_sim)
holiday_hourly = hourly_exposure(holiday_sim)
weekday_link_hour = link_hour_matrix(weekday_sim)
holiday_link_hour = link_hour_matrix(holiday_sim)
weekday_geom = link_geometries(weekday_sim)
holiday_geom = link_geometries(holiday_sim)

print("weekday total exposure", weekday_hourly.sum())
print("holiday total exposure", holiday_hourly.sum() if not holiday_hourly.empty else "missing")
print("weekday link-hour matrix", weekday_link_hour.shape)
print("geometry available", weekday_geom is not None)


weekday total exposure 38970306
holiday total exposure 32129418
weekday link-hour matrix (3887, 24)
geometry available True


## 3. Accident summaries and temporal validation against simulated exposure

In [24]:
# 8a_accident_overview.png - yearly accidents and road-category distribution.
try:
    hourly_acc = acc["hourly"]
    road_cat = acc["road_category"]

    yearly = hourly_acc.groupby("year")[["incidents", "deaths", "injured"]].sum().reset_index()
    road_cat_no_total = road_cat[road_cat["road_category"] != "Totale"].copy()

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), constrained_layout=True)

    ax = axes[0]
    ax.plot(yearly["year"], yearly["incidents"], marker="o", linewidth=2.2,
            color=ACCIDENT_COLORS["incidents"], label="Incidents")
    ax2 = ax.twinx()
    ax2.plot(yearly["year"], yearly["deaths"], marker="s", linewidth=2.0,
             color=ACCIDENT_COLORS["deaths"], label="Deaths")
    ax.set_xlabel("Year")
    ax.set_ylabel("Incidents")
    ax2.set_ylabel("Deaths")
    ax.set_xticks(yearly["year"])
    clean_axis(ax)
    ax2.spines["top"].set_visible(False)
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc="upper left", frameon=True)

    ax = axes[1]
    pivot = road_cat_no_total.pivot_table(index="year", columns="road_category", values="incidents", aggfunc="sum")
    shares = pivot.div(pivot.sum(axis=1), axis=0) * 100
    bottom = np.zeros(len(shares))
    colors = ["#8D8D8D", "#D08C60", "#6B8FA6"]
    for color, col in zip(colors, shares.columns):
        ax.bar(shares.index.astype(str), shares[col], bottom=bottom, color=color, label=col,
               edgecolor="white", linewidth=0.8)
        bottom += shares[col].values
    ax.set_ylabel("Share of incidents (%)")
    ax.set_xlabel("Year")
    ax.set_ylim(0, 100)
    clean_axis(ax)
    ax.legend(loc="upper right", frameon=True)

    save_fig_local(fig, "8a_accident_overview")
    plt.show()
except Exception as exc:
    print(f"ERROR 8a_accident_overview: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/8a_accident_overview.png
Saved: figures_incident_risk/8a_accident_overview.pdf


/tmp/ipykernel_10043/864225377.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# 8b_hourly_accidents_vs_exposure.png - hourly accidents versus simulated exposure.
try:
    hourly_acc_mean = acc["hourly"].groupby("hour")["incidents"].mean().reindex(range(24), fill_value=0.0)
    hourly_acc_share = hourly_acc_mean / hourly_acc_mean.sum()

    weekday_share = weekday_hourly / weekday_hourly.sum()
    holiday_share = holiday_hourly / holiday_hourly.sum() if not holiday_hourly.empty and holiday_hourly.sum() > 0 else None

    x = np.arange(24)
    fig, ax = plt.subplots(figsize=(9.5, 4.8), constrained_layout=True)

    ax.plot(x, hourly_acc_share.values * 100, color=ACCIDENT_COLORS["incidents"],
            marker="o", linewidth=2.2, markersize=4.5, label="Accidents, 2022--2024 mean")
    ax.plot(x, weekday_share.values * 100, color=ACCIDENT_COLORS["weekday"],
            marker="s", linewidth=2.0, markersize=4.2, label="Weekday simulated exposure")

    if holiday_share is not None:
        ax.plot(x, holiday_share.values * 100, color=ACCIDENT_COLORS["holiday"],
                linestyle=(0, (4, 2)), marker="^", markevery=2, linewidth=2.0,
                label="Holiday simulated exposure")

#     ax.set_title("Hourly accident distribution and simulated traffic exposure", loc="left", fontweight="bold")
    ax.set_ylabel("Daily share (%)")
    ax.set_xlabel("Time")
    ax.set_xticks(HOUR_TICKS)
    ax.set_xticklabels([HOUR_LABELS[i] for i in HOUR_TICKS], rotation=45, ha="right")
    clean_axis(ax)
    ax.legend(loc="upper left", frameon=True, ncol=1)

    save_fig_local(fig, "8b_hourly_accidents_vs_exposure")
    plt.show()
except Exception as exc:
    print(f"ERROR 8b_hourly_accidents_vs_exposure: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/8b_hourly_accidents_vs_exposure.png
Saved: figures_incident_risk/8b_hourly_accidents_vs_exposure.pdf


/tmp/ipykernel_10043/529919131.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# 8c_exposure_normalized_hourly_risk.png - risk intensity normalized by simulated exposure.
try:
    hourly_acc_mean = acc["hourly"].groupby("hour")["incidents"].mean().reindex(range(24), fill_value=0.0)
    hourly_acc_share = hourly_acc_mean / hourly_acc_mean.sum()

    def relative_hourly_risk(exposure):
        exposure_share = exposure / exposure.sum()
        risk = hourly_acc_share / exposure_share.replace(0, np.nan)
        return risk / risk.replace([np.inf, -np.inf], np.nan).mean()

    weekday_risk = relative_hourly_risk(weekday_hourly)
    holiday_risk = relative_hourly_risk(holiday_hourly) if not holiday_hourly.empty and holiday_hourly.sum() > 0 else None

    x = np.arange(24)
    fig, ax = plt.subplots(figsize=(9.5, 4.8), constrained_layout=True)

    ax.axhline(1.0, color="#555555", linestyle="--", linewidth=1.2, alpha=0.8, label="Daily average risk")
    ax.bar(x - 0.18, weekday_risk.values, width=0.36, color=ACCIDENT_COLORS["weekday"],
           edgecolor="white", linewidth=0.6, label="Using weekday exposure", zorder=3)
    if holiday_risk is not None:
        ax.bar(x + 0.18, holiday_risk.values, width=0.36, color=ACCIDENT_COLORS["holiday"],
               edgecolor="white", linewidth=0.6, label="Using holiday exposure", zorder=3)

    # ax.set_title("Exposure-normalized hourly accident intensity", loc="left", fontweight="bold")
    ax.set_ylabel("Relative risk index")
    ax.set_xlabel("Time")
    ax.set_xticks(HOUR_TICKS)
    ax.set_xticklabels([HOUR_LABELS[i] for i in HOUR_TICKS], rotation=45, ha="right")
    ax.set_ylim(0, max(2.5, np.nanmax(weekday_risk.values) * 1.15))
    clean_axis(ax)
    ax.legend(loc="upper right", frameon=True)

    save_fig_local(fig, "8c_exposure_normalized_hourly_risk")
    plt.show()
except Exception as exc:
    print(f"ERROR 8c_exposure_normalized_hourly_risk: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/8c_exposure_normalized_hourly_risk.png
Saved: figures_incident_risk/8c_exposure_normalized_hourly_risk.pdf


/tmp/ipykernel_10043/3902259995.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Seasonal exposure and road-category accident structure

In [27]:
# 8d_seasonal_exposure_vs_accidents.png - seasonal traffic exposure and accident seasonality.
try:
    monthly = acc["monthly"].copy()
    incident_season = monthly.groupby("season")["incidents"].sum()
    incident_season_share = incident_season / incident_season.sum() * 100

    weekday_season = seasonal_exposure(weekday_sim)
    if weekday_season.empty or weekday_season["season"].nunique() <= 1:
        raise ValueError("Seasonal simulation exposure not available. Expected a 'season' column with multiple seasons.")

    weekday_season_share = weekday_season.set_index("season")["counter"]
    weekday_season_share = weekday_season_share / weekday_season_share.sum() * 100

    season_order = ["winter", "spring", "summer", "autumn"]
    incident_season_share = incident_season_share.reindex(season_order)
    weekday_season_share = weekday_season_share.reindex(season_order)

    fig, ax = plt.subplots(figsize=(8.4, 4.8), constrained_layout=True)
    x = np.arange(len(season_order))
    width = 0.36

    ax.bar(x - width/2, incident_season_share.values, width=width,
           color=ACCIDENT_COLORS["incidents"], edgecolor="white", linewidth=0.8,
           label="Accidents, 2022--2024")
    ax.bar(x + width/2, weekday_season_share.values, width=width,
           color=ACCIDENT_COLORS["exposure"], edgecolor="white", linewidth=0.8,
           label="Weekday simulated exposure")

    ax.set_ylabel("Annual share (%)")
    ax.set_xlabel("Season")
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in season_order])
    clean_axis(ax)
    ax.legend(loc="upper left", frameon=True)

    save_fig_local(fig, "8d_seasonal_exposure_vs_accidents")
    plt.show()
except Exception as exc:
    print(f"SKIPPED 8d_seasonal_exposure_vs_accidents: {type(exc).__name__}: {exc}")


Saved: figures_incident_risk/8d_seasonal_exposure_vs_accidents.png
Saved: figures_incident_risk/8d_seasonal_exposure_vs_accidents.pdf


/tmp/ipykernel_10043/1881722545.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Hazard-weighted exposure on the road network

The Excel tables do not provide point locations for crashes. To still connect accidents and simulated links, we compute a **temporal hazard-weighted exposure**:

\[
H_i = \sum_h F_i(h) R(h),
\]

where \(F_i(h)\) is the simulated flow on road segment \(i\), and \(R(h)\) is the exposure-normalized accident intensity by hour. This does not locate crashes; it highlights which simulated links carry traffic during hours with higher regional accident intensity.

In [28]:
# Helper for hazard-weighted link exposure.

def compute_hazard_weighted_exposure(link_hour, exposure_hourly, hourly_accidents):
    if link_hour.empty:
        return pd.Series(dtype=float)
    acc_share = hourly_accidents / hourly_accidents.sum()
    exp_share = exposure_hourly / exposure_hourly.sum()
    risk = acc_share / exp_share.replace(0, np.nan)
    risk = risk.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    risk = risk / risk[risk > 0].mean()
    risk = risk.reindex(range(24), fill_value=1.0)
    return (link_hour.reindex(columns=range(24), fill_value=0.0) * risk.values).sum(axis=1)

hourly_acc_mean = acc["hourly"].groupby("hour")["incidents"].mean().reindex(range(24), fill_value=0.0)
weekday_hazard = compute_hazard_weighted_exposure(weekday_link_hour, weekday_hourly, hourly_acc_mean)
holiday_hazard = compute_hazard_weighted_exposure(holiday_link_hour, holiday_hourly, hourly_acc_mean) if not holiday_link_hour.empty else pd.Series(dtype=float)

hazard_summary = pd.DataFrame({
    "weekday_hazard": weekday_hazard,
    "weekday_total_flow": weekday_link_hour.sum(axis=1) if not weekday_link_hour.empty else pd.Series(dtype=float),
})
if not holiday_hazard.empty:
    hazard_summary["holiday_hazard"] = holiday_hazard
    hazard_summary["holiday_total_flow"] = holiday_link_hour.sum(axis=1)

hazard_summary.to_csv(OUTPUT_DIR / "hazard_weighted_exposure_summary.csv")
display(hazard_summary.head())
print(f"Saved summary: {OUTPUT_DIR / 'hazard_weighted_exposure_summary.csv'}")


,weekday_hazard,weekday_total_flow,holiday_hazard,holiday_total_flow
poly_lid,,,,
0,900.995267,1194,792.778280,1047
1,2960.894988,3846,2865.550537,3614
2,3086.379056,3944,2425.515638,3188
3,3985.626930,5297,3301.379506,4462
4,5873.583646,7468,4426.671059,5783


Saved summary: figures_incident_risk/hazard_weighted_exposure_summary.csv


In [29]:
# 8e_hazard_weighted_exposure_map.png - map of weekday hazard-weighted exposure.
try:
    if weekday_geom is None:
        raise ValueError("No geometry available in the simulation table. Use aggregated feriali.csv/festivi.csv with WKT geometry.")

    map_df = weekday_geom.merge(
        hazard_summary.reset_index().rename(columns={"index": "poly_lid"}),
        on="poly_lid",
        how="left",
    )
    map_df["weekday_hazard"] = map_df["weekday_hazard"].fillna(0.0)
    map_web = map_df.to_crs(epsg=3857)

    active = map_web[map_web["weekday_hazard"] > 0]
    vmin = max(1, active["weekday_hazard"].quantile(0.05)) if not active.empty else 1
    vmax = active["weekday_hazard"].quantile(0.98) if not active.empty else 10

    fig, ax = plt.subplots(figsize=(10.5, 6.6), constrained_layout=True)
    map_web.plot(ax=ax, color=PALETTE["roads_minor"], linewidth=0.42, alpha=0.48, zorder=1)
    active.plot(ax=ax, column="weekday_hazard", cmap=risk_cmap, linewidth=1.15,
                norm=LogNorm(vmin=vmin, vmax=max(vmax, vmin + 1)), zorder=3)

    if cx is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False, alpha=0.72, zorder=0)
            except Exception as exc:
                print(f"Basemap unavailable: {exc}")

    sm = plt.cm.ScalarMappable(cmap=risk_cmap, norm=LogNorm(vmin=vmin, vmax=max(vmax, vmin + 1)))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.74, pad=0.012)
    cbar.set_label("Hazard-weighted simulated exposure", fontsize=15, fontweight="bold")
    cbar.ax.tick_params(labelsize=13)

    ax.set_axis_off()

    save_fig_local(fig, "8e_hazard_weighted_exposure_map")
    plt.show()
except Exception as exc:
    print(f"SKIPPED 8e_hazard_weighted_exposure_map: {type(exc).__name__}: {exc}")


Saved: figures_incident_risk/8e_hazard_weighted_exposure_map.png
Saved: figures_incident_risk/8e_hazard_weighted_exposure_map.pdf


/tmp/ipykernel_10043/2873469649.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
# 8f_weekday_holiday_hazard_difference.png - difference map when holiday simulations are available.
try:
    if weekday_geom is None:
        raise ValueError("No geometry available in the simulation table.")
    if holiday_hazard.empty:
        raise ValueError("Holiday simulation not available.")

    diff = pd.DataFrame({
        "poly_lid": sorted(set(weekday_hazard.index) | set(holiday_hazard.index))
    })
    diff["weekday_hazard"] = diff["poly_lid"].map(weekday_hazard).fillna(0.0)
    diff["holiday_hazard"] = diff["poly_lid"].map(holiday_hazard).fillna(0.0)
    diff["relative_difference"] = (
        (diff["weekday_hazard"] - diff["holiday_hazard"]) /
        (diff["weekday_hazard"] + diff["holiday_hazard"]).replace(0, np.nan)
    ).fillna(0.0)

    map_df = weekday_geom.merge(diff, on="poly_lid", how="left").fillna({"relative_difference": 0.0})
    map_web = map_df.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10.5, 6.6), constrained_layout=True)
    map_web.plot(ax=ax, color=PALETTE["roads_minor"], linewidth=0.40, alpha=0.38, zorder=1)
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    map_web.plot(ax=ax, column="relative_difference", cmap="RdBu_r", norm=norm,
                 linewidth=1.15, alpha=0.95, zorder=3)

    if cx is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False, alpha=0.72, zorder=0)
            except Exception as exc:
                print(f"Basemap unavailable: {exc}")

    sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.74, pad=0.012)
    cbar.set_label("(Weekday - holiday) / total hazard exposure", fontsize=15, fontweight="bold")
    cbar.ax.tick_params(labelsize=13)

    ax.set_axis_off()

    save_fig_local(fig, "8f_weekday_holiday_hazard_difference")
    plt.show()
except Exception as exc:
    print(f"SKIPPED 8f_weekday_holiday_hazard_difference: {type(exc).__name__}: {exc}")


Saved: figures_incident_risk/8f_weekday_holiday_hazard_difference.png
Saved: figures_incident_risk/8f_weekday_holiday_hazard_difference.pdf


/tmp/ipykernel_10043/4020315256.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
# ==========================================================
# 9. Preparation for advanced incident-risk analyses
# ----------------------------------------------------------
# This cell defines the variables used by the next four cells:
# - hourly_compare
# - hourly_risk_mean
# - road_exposure_gdf
# It only uses variables already created in this notebook.
# ==========================================================

try:
    if gpd is None:
        raise ImportError("geopandas is not available.")
    if weekday_geom is None:
        raise ValueError(
            "weekday_geom is not available. The simulation table must contain a WKT 'geometry' column, "
            "as in feriali/feriali.csv."
        )
    if weekday_sim.empty:
        raise ValueError("weekday_sim is empty.")
    if weekday_hourly.empty or weekday_hourly.sum() <= 0:
        raise ValueError("weekday_hourly exposure is empty or zero.")

    HOURS = np.arange(24)

    # ------------------------------------------------------
    # Hourly accident distribution, averaged over 2022--2024
    # ------------------------------------------------------
    hourly_acc_mean = (
        acc["hourly"]
        .groupby("hour")["incidents"]
        .mean()
        .reindex(HOURS, fill_value=0.0)
        .astype(float)
    )

    accident_share = hourly_acc_mean / hourly_acc_mean.sum()

    # ------------------------------------------------------
    # Weekday/holiday simulated exposure shares
    # ------------------------------------------------------
    weekday_exposure = weekday_hourly.reindex(HOURS, fill_value=0.0).astype(float)
    weekday_exposure_share = weekday_exposure / weekday_exposure.sum()

    has_holiday = (
        isinstance(holiday_hourly, pd.Series)
        and not holiday_hourly.empty
        and holiday_hourly.sum() > 0
    )

    if has_holiday:
        holiday_exposure = holiday_hourly.reindex(HOURS, fill_value=0.0).astype(float)
        holiday_exposure_share = holiday_exposure / holiday_exposure.sum()
    else:
        holiday_exposure = pd.Series(np.nan, index=HOURS)
        holiday_exposure_share = pd.Series(np.nan, index=HOURS)

    # ------------------------------------------------------
    # Exposure-normalized risk profiles
    # risk = accident share / exposure share, normalized to mean 1
    # ------------------------------------------------------
    def normalized_risk(acc_share, exp_share):
        risk = acc_share / exp_share.replace(0, np.nan)
        risk = risk.replace([np.inf, -np.inf], np.nan)
        mean_positive = risk[risk > 0].mean()
        if pd.isna(mean_positive) or mean_positive == 0:
            mean_positive = 1.0
        return (risk / mean_positive).fillna(1.0)

    risk_weekday = normalized_risk(accident_share, weekday_exposure_share)

    if has_holiday:
        risk_holiday = normalized_risk(accident_share, holiday_exposure_share)
    else:
        risk_holiday = pd.Series(np.nan, index=HOURS)

    hourly_compare = pd.DataFrame({
        "hour": HOURS,
        "accidents": hourly_acc_mean.values,
        "accident_share": accident_share.values,
        "weekday_exposure": weekday_exposure.values,
        "weekday_exposure_share": weekday_exposure_share.values,
        "risk_weekday": risk_weekday.values,
        "holiday_exposure": holiday_exposure.values,
        "holiday_exposure_share": holiday_exposure_share.values,
        "risk_holiday": risk_holiday.values,
    })

    # This is the risk profile used to weight every link-hour exposure.
    hourly_risk_mean = pd.Series(
        hourly_compare["risk_weekday"].values,
        index=hourly_compare["hour"].astype(int),
        name="hazard_factor",
    )

    # ------------------------------------------------------
    # Build long-format link-hour GeoDataFrame from weekday simulation
    # ------------------------------------------------------
    exposure_cols = ["poly_lid", "hour", "counter"]
    if "season" in weekday_sim.columns:
        exposure_cols.append("season")

    weekday_long = weekday_sim[exposure_cols].copy()
    weekday_long = weekday_long.dropna(subset=["poly_lid", "hour"])
    weekday_long["poly_lid"] = weekday_long["poly_lid"].astype(weekday_geom["poly_lid"].dtype)
    weekday_long["hour"] = weekday_long["hour"].astype(int)
    weekday_long["counter"] = pd.to_numeric(weekday_long["counter"], errors="coerce").fillna(0.0)

    road_exposure_gdf = weekday_geom.merge(
        weekday_long,
        on="poly_lid",
        how="inner",
    )

    road_exposure_gdf["hazard_factor"] = (
        road_exposure_gdf["hour"]
        .map(hourly_risk_mean)
        .fillna(hourly_risk_mean.mean())
    )

    road_exposure_gdf["hazard_weighted_exposure"] = (
        road_exposure_gdf["counter"].astype(float)
        * road_exposure_gdf["hazard_factor"].astype(float)
    )

    print("Created hourly_compare:", hourly_compare.shape)
    print("Created road_exposure_gdf:", road_exposure_gdf.shape)
    print("Unique road links:", road_exposure_gdf["poly_lid"].nunique())
    print("Hour range:", sorted(road_exposure_gdf["hour"].dropna().unique())[:3], "...", sorted(road_exposure_gdf["hour"].dropna().unique())[-3:])

    display(hourly_compare.head())
    display(
        road_exposure_gdf[
            ["poly_lid", "hour", "counter", "hazard_factor", "hazard_weighted_exposure"]
        ].head()
    )

except Exception as exc:
    print(f"ERROR preparing advanced risk variables: {type(exc).__name__}: {exc}")
    raise


Created hourly_compare: (24, 9)
Created road_exposure_gdf: (373152, 7)
Unique road links: 3887
Hour range: [np.int64(0), np.int64(1), np.int64(2)] ... [np.int64(21), np.int64(22), np.int64(23)]


,hour,accidents,accident_share,weekday_exposure,weekday_exposure_share,risk_weekday,holiday_exposure,holiday_exposure_share,risk_holiday
0,0,244.666667,0.014609,128747.0,0.003304,3.437260,235110.0,0.007318,1.527819
1,1,167.666667,0.010011,128747.0,0.003304,2.355507,235110.0,0.007318,1.046993
2,2,113.000000,0.006747,226427.0,0.005810,0.902662,526556.0,0.016389,0.315066
3,3,106.000000,0.006329,176204.0,0.004521,1.088090,373615.0,0.011628,0.416533
4,4,110.333333,0.006588,173549.0,0.004453,1.149898,268481.0,0.008356,0.603339


,poly_lid,hour,counter,hazard_factor,hazard_weighted_exposure
0,0,0,0,3.437260,0.000000
1,0,1,0,2.355507,0.000000
2,0,2,3,0.902662,2.707985
3,0,3,2,1.088090,2.176180
4,0,4,1,1.149898,1.149898


In [32]:
# 9a_risk_centrality.png - Geolocate links ranked by hazard-weighted exposure centrality.
try:
    risk_gdf = road_exposure_gdf.copy()

    ranking = (
        risk_gdf
        .groupby("poly_lid", as_index=False)
        .agg(
            total_exposure=("counter", "sum"),
            hazard_weighted_exposure=("hazard_weighted_exposure", "sum"),
            mean_hazard_factor=("hazard_factor", "mean"),
            geometry=("geometry", "first"),
        )
    )

    total_hazard = ranking["hazard_weighted_exposure"].sum()
    if total_hazard <= 0:
        raise ValueError("Total hazard-weighted exposure is zero.")

    ranking["risk_centrality"] = ranking["hazard_weighted_exposure"] / total_hazard
    ranking = ranking.sort_values("risk_centrality", ascending=False)

    top = ranking.head(20).copy()
    top["poly_lid_label"] = top["poly_lid"].astype(str)

    ranking_gdf = gpd.GeoDataFrame(
        ranking,
        geometry="geometry",
        crs=road_exposure_gdf.crs,
    )
    map_gdf = ranking_gdf.to_crs(epsg=3857)
    top_gdf = ranking_gdf.head(20).to_crs(epsg=3857)
    label_gdf = ranking_gdf.head(6).to_crs(epsg=3857)

    fig, (ax_map, ax_rank) = plt.subplots(
        1,
        2,
        figsize=(15.2, 7.0),
        gridspec_kw={"width_ratios": [1.45, 0.95]},
        constrained_layout=True,
    )

    map_gdf.plot(
        ax=ax_map,
        color=PALETTE["roads_minor"],
        linewidth=0.45,
        alpha=0.34,
        zorder=1,
    )
    top_gdf.plot(
        ax=ax_map,
        column="risk_centrality",
        cmap=risk_cmap,
        linewidth=3.2,
        alpha=0.96,
        legend=True,
        zorder=4,
        legend_kwds={
            "label": "Risk centrality",
            "shrink": 0.70,
        },
    )

    if cx is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                cx.add_basemap(
                    ax_map,
                    source=cx.providers.CartoDB.Positron,
                    attribution=False,
                    alpha=0.65,
                    zorder=0,
                )
            except Exception as exc:
                print(f"Basemap unavailable: {exc}")

    for _, row in label_gdf.iterrows():
        point = row.geometry.interpolate(0.5, normalized=True)
        ax_map.annotate(
            str(row["poly_lid"]),
            xy=(point.x, point.y),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=14,
            fontweight="bold",
            color=PALETTE["text"],
            bbox={"boxstyle": "round,pad=0.22", "fc": "white", "ec": "#DDDDDD", "alpha": 0.92},
            zorder=5,
        )

    top10 = top.head(10).copy()
    ax_rank.barh(
        top10["poly_lid_label"][::-1],
        100 * top10["risk_centrality"][::-1],
        color=ACCIDENT_COLORS["risk"],
        edgecolor="white",
        linewidth=1.0,
        zorder=3,
    )
    ax_rank.set_xlabel("Share of hazard-weighted exposure (%)", fontsize=18)
    ax_rank.set_ylabel("Polyline ID", fontsize=18)
    ax_rank.tick_params(axis="both", labelsize=14)
    ax_rank.grid(axis="x", color=PALETTE["grid"], alpha=0.7, zorder=0)
    ax_rank.set_axisbelow(True)
    ax_rank.spines["top"].set_visible(False)
    ax_rank.spines["right"].set_visible(False)

    ax_map.set_axis_off()

    if fig.axes:
        for extra_ax in fig.axes:
            if extra_ax not in (ax_map, ax_rank):
                extra_ax.tick_params(labelsize=13)
                extra_ax.set_ylabel("Risk centrality", fontsize=15, fontweight="bold")

    save_fig_local(fig, "9a_risk_centrality")

    ranking.to_csv(OUTPUT_DIR / "risk_centrality_ranking.csv", index=False)
    print(f"Saved: {OUTPUT_DIR / 'risk_centrality_ranking.csv'}")
    display(ranking.head(20)[["poly_lid", "total_exposure", "hazard_weighted_exposure", "risk_centrality"]])

except Exception as exc:
    print(f"ERROR 9a_risk_centrality: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/9a_risk_centrality.png
Saved: figures_incident_risk/9a_risk_centrality.pdf
Saved: figures_incident_risk/risk_centrality_ranking.csv


,poly_lid,total_exposure,hazard_weighted_exposure,risk_centrality
2840,2861,112330,89033.563540,0.002939
2317,2335,104655,84621.434681,0.002793
378,382,110501,83057.127134,0.002742
1526,1535,106953,81475.882086,0.002690
2372,2390,94024,77814.076591,0.002569
1982,1994,93683,76477.707844,0.002525
1152,1159,95212,75535.789290,0.002494
3653,3687,92611,75383.378396,0.002489
145,146,88830,72893.301212,0.002406
2986,3009,93423,71821.472042,0.002371


In [33]:
# 9b_exposure_elasticity.png - Scaling between simulated exposure and accident intensity by hour.
try:
    elasticity_df = hourly_compare.copy()
    elasticity_df = elasticity_df.replace([np.inf, -np.inf], np.nan)

    elasticity_df = elasticity_df.dropna(
        subset=["accident_share", "weekday_exposure_share"]
    )

    elasticity_df = elasticity_df[
        (elasticity_df["accident_share"] > 0)
        & (elasticity_df["weekday_exposure_share"] > 0)
    ].copy()

    if len(elasticity_df) < 3:
        raise ValueError("Not enough valid hourly points for a log-log fit.")

    x_log = np.log(elasticity_df["weekday_exposure_share"].values)
    y_log = np.log(elasticity_df["accident_share"].values)

    slope, intercept = np.polyfit(x_log, y_log, 1)
    r = np.corrcoef(x_log, y_log)[0, 1]
    r2 = r**2

    fig, ax = plt.subplots(figsize=(8.5, 7.0), constrained_layout=True)

    ax.scatter(
        elasticity_df["weekday_exposure_share"],
        elasticity_df["accident_share"],
        s=95,
        color=ACCIDENT_COLORS["incidents"],
        edgecolor="white",
        linewidth=0.8,
        alpha=0.92,
        zorder=3,
    )

    xx = np.geomspace(
        elasticity_df["weekday_exposure_share"].min(),
        elasticity_df["weekday_exposure_share"].max(),
        200,
    )
    yy = np.exp(intercept) * xx**slope

    ax.plot(
        xx,
        yy,
        color=PALETTE["text"],
        linewidth=2.3,
        linestyle="--",
        label=rf"$I \sim E^{{{slope:.2f}}}$",
        zorder=4,
    )

    for _, row in elasticity_df.iterrows():
        ax.text(
            row["weekday_exposure_share"],
            row["accident_share"],
            f"{int(row['hour']):02d}",
            fontsize=12,
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
            zorder=5,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")

    from matplotlib.ticker import LogLocator, FuncFormatter
    _log_fmt = FuncFormatter(lambda v, _: f"{v:.3g}")
    for _axis in (ax.xaxis, ax.yaxis):
        _axis.set_major_formatter(_log_fmt)
        _axis.set_minor_locator(LogLocator(base=10.0, subs=[2, 3, 5], numticks=30))
        _axis.set_minor_formatter(_log_fmt)
    ax.tick_params(axis="both", which="minor", labelsize=10)
    ax.tick_params(axis="both", which="major", labelsize=13)

    ax.set_xlabel("Simulated weekday exposure share", fontsize=18)
    ax.set_ylabel("Accident share", fontsize=18)

    ax.text(
        0.04,
        0.96,
        rf"Elasticity $\alpha$ = {slope:.2f}" + "\n" + rf"$R^2$ = {r2:.2f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=18,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="#DDDDDD",
            alpha=0.92,
        ),
    )

    ax.grid(True, color=PALETTE["grid"], alpha=0.65)
    ax.legend(loc="lower right", fontsize=17)

    save_fig_local(fig, "9b_exposure_elasticity")

    print(f"Elasticity alpha = {slope:.3f}; R2 = {r2:.3f}")

except Exception as exc:
    print(f"ERROR 9b_exposure_elasticity: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/9b_exposure_elasticity.png
Saved: figures_incident_risk/9b_exposure_elasticity.pdf
Elasticity alpha = 0.718; R2 = 0.852


In [34]:
# 9c_hourly_risk_clusters.png - Cluster hours by exposure-normalized risk profile.
try:
    from scipy.cluster.hierarchy import linkage, fcluster

    cluster_df = hourly_compare.copy()

    candidate_features = [
        "accident_share",
        "weekday_exposure_share",
        "risk_weekday",
    ]

    # Use holiday features only if the holiday simulation exists.
    if "holiday_exposure_share" in cluster_df.columns and cluster_df["holiday_exposure_share"].notna().any():
        candidate_features.append("holiday_exposure_share")
    if "risk_holiday" in cluster_df.columns and cluster_df["risk_holiday"].notna().any():
        candidate_features.append("risk_holiday")

    features = cluster_df[candidate_features].replace([np.inf, -np.inf], np.nan)
    features = features.fillna(features.median(numeric_only=True))

    X = features.values.astype(float)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    X = (X - X.mean(axis=0)) / std

    Z = linkage(X, method="ward")

    n_clusters = 3
    cluster_df["risk_cluster"] = fcluster(Z, n_clusters, criterion="maxclust")

    cluster_palette = {
        1: ACCIDENT_COLORS["incidents"],
        2: ACCIDENT_COLORS["weekday"],
        3: ACCIDENT_COLORS["holiday"],
    }

    fig, ax = plt.subplots(figsize=(11.5, 6.5), constrained_layout=True)

    for cluster, group in cluster_df.groupby("risk_cluster"):
        color = cluster_palette.get(int(cluster), PALETTE["text"])

        ax.scatter(
            group["hour"],
            group["risk_weekday"],
            s=125,
            color=color,
            edgecolor="white",
            linewidth=1.1,
            label=f"Risk cluster {int(cluster)}",
            zorder=4,
        )

        ax.plot(
            group["hour"],
            group["risk_weekday"],
            color=color,
            linewidth=2.8,
            alpha=0.70,
            zorder=3,
        )

    ax.axhline(
        1.0,
        color=PALETTE["text"],
        linestyle="--",
        linewidth=1.8,
        alpha=0.75,
        label="Daily average risk",
    )

    ax.set_xlabel("Time", fontsize=18)
    ax.set_ylabel("Relative risk index", fontsize=18)
    ax.set_xticks(HOUR_TICKS)
    ax.set_xticklabels([HOUR_LABELS[h] for h in HOUR_TICKS], rotation=45, ha="right")
    ax.grid(True, axis="y", color=PALETTE["grid"], alpha=0.7)
    ax.legend(loc="upper right", fontsize=17)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    save_fig_local(fig, "9c_hourly_risk_clusters")

    cluster_df.to_csv(OUTPUT_DIR / "hourly_risk_clusters.csv", index=False)
    print(f"Saved: {OUTPUT_DIR / 'hourly_risk_clusters.csv'}")
    display(cluster_df[["hour", "risk_cluster", "risk_weekday", "risk_holiday"]])

except Exception as exc:
    print(f"ERROR 9c_hourly_risk_clusters: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/9c_hourly_risk_clusters.png
Saved: figures_incident_risk/9c_hourly_risk_clusters.pdf
Saved: figures_incident_risk/hourly_risk_clusters.csv


,hour,risk_cluster,risk_weekday,risk_holiday
0,0,1,3.437260,1.527819
1,1,1,2.355507,1.046993
2,2,1,0.902662,0.315066
3,3,1,1.088090,0.416533
4,4,1,1.149898,0.603339
5,5,1,1.322950,1.210135
6,6,1,1.002083,1.907733
7,7,3,1.149612,4.200444
8,8,3,0.680896,2.710307
9,9,3,0.566569,1.201531


In [35]:
# 9d_high_risk_anomaly_map.png - Map links with high hazard relative to exposure.
try:
    anomaly_gdf = road_exposure_gdf.copy()

    link_stats = (
        anomaly_gdf
        .groupby("poly_lid", as_index=False)
        .agg(
            exposure=("counter", "sum"),
            hazard=("hazard_weighted_exposure", "sum"),
            mean_hazard_factor=("hazard_factor", "mean"),
            geometry=("geometry", "first"),
        )
    )

    link_stats = gpd.GeoDataFrame(
        link_stats,
        geometry="geometry",
        crs=road_exposure_gdf.crs,
    )

    link_stats = link_stats[link_stats["exposure"] > 0].copy()

    link_stats["exposure_rank"] = link_stats["exposure"].rank(pct=True)
    link_stats["hazard_rank"] = link_stats["hazard"].rank(pct=True)
    link_stats["risk_anomaly"] = link_stats["hazard_rank"] - link_stats["exposure_rank"]

    # High anomaly = much higher hazard rank than exposure rank.
    high_anomaly = link_stats[
        link_stats["risk_anomaly"] >= link_stats["risk_anomaly"].quantile(0.95)
    ].copy()

    map_gdf = link_stats.to_crs(epsg=3857)
    high_anomaly = high_anomaly.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(13.0, 7.2), constrained_layout=True)

    map_gdf.plot(
        ax=ax,
        color=PALETTE["roads_minor"],
        linewidth=0.55,
        alpha=0.38,
        zorder=2,
    )

    if not high_anomaly.empty:
        high_anomaly.plot(
            ax=ax,
            column="risk_anomaly",
            cmap="Reds",
            linewidth=3.4,
            alpha=0.96,
            legend=True,
            zorder=4,
            legend_kwds={
                "label": "Hazard rank - exposure rank",
                "shrink": 0.72,
            },
        )

    if cx is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                cx.add_basemap(
                    ax,
                    source=cx.providers.CartoDB.Positron,
                    attribution=False,
                    alpha=0.65,
                    zorder=0,
                )
            except Exception as exc:
                print(f"Basemap unavailable: {exc}")

    ax.set_axis_off()

    save_fig_local(fig, "9d_high_risk_anomaly_map")

    link_stats.to_csv(OUTPUT_DIR / "high_risk_anomaly_links.csv", index=False)
    print(f"Saved: {OUTPUT_DIR / 'high_risk_anomaly_links.csv'}")

    display(
        link_stats
        .sort_values("risk_anomaly", ascending=False)
        .head(20)
        [["poly_lid", "exposure", "hazard", "mean_hazard_factor", "risk_anomaly"]]
    )

except Exception as exc:
    print(f"ERROR 9d_high_risk_anomaly_map: {type(exc).__name__}: {exc}")
    raise


Saved: figures_incident_risk/9d_high_risk_anomaly_map.png
Saved: figures_incident_risk/9d_high_risk_anomaly_map.pdf
Saved: figures_incident_risk/high_risk_anomaly_links.csv


,poly_lid,exposure,hazard,mean_hazard_factor,risk_anomaly
281,282,4450,6884.496571,1.0,0.170999
117,118,13229,22476.803170,1.0,0.166928
1491,1500,5357,7775.246033,1.0,0.162543
278,279,14410,23473.597205,1.0,0.149702
123,124,2268,3330.100659,1.0,0.146884
707,711,4862,6676.602264,1.0,0.146257
280,281,2447,3390.161255,1.0,0.132790
851,855,6259,7767.198542,1.0,0.127466
857,861,3415,4356.716692,1.0,0.111494
2028,2041,2455,2986.511164,1.0,0.101942
